# Frontier-Model Reasoning Trace Browser

Browse and examine captured reasoning traces from frontier models on `equation_numeric_guess` and `bit_manipulation` puzzles.

**Run all cells top-to-bottom** — the summary table and browser function will be ready after Cell 2.

In [1]:
# ── Cell 1: LOADER ──────────────────────────────────────────────────────────
# Globs every results.jsonl under bedrock/**/  plus the special
# eqguess_capture/deepseek_existing.jsonl file, tags each row with the
# source model name, then joins the problem prompt from train.csv.

import json
import glob
import os
import re
import textwrap
import warnings
from pathlib import Path

import pandas as pd

warnings.filterwarnings("ignore")

# ── Repo root: two levels up from this notebook ──────────────────────────────
REPO = Path(os.path.abspath("__file__" if "__file__" in dir() else ".")).parent.parent.parent
# Fallback: try known absolute path
if not (REPO / "foundation").exists():
    REPO = Path("/home/ec2-user/dev/nemotron_training")

BEDROCK  = REPO / "bedrock"
TRAIN_CSV = REPO / "foundation" / "competition" / "train.csv"

# ── Helper: derive a short human-readable model label from a file path ───────
def _model_label(path: Path) -> str:
    """Turn  bedrock/eqguess_capture/eq_gptoss/results.jsonl  →  'gptoss'
    and    bedrock/bitman_capture/bm_deepseek/results.jsonl  →  'deepseek'
    and    bedrock/eqguess_capture/deepseek_existing.jsonl   →  'deepseek'
    and    bedrock/gptoss_120b/results.jsonl                 →  'gptoss_120b'
    """
    parts = path.parts
    # The file is named deepseek_existing.jsonl directly (not results.jsonl)
    if path.name != "results.jsonl":
        stem = path.stem  # e.g. 'deepseek_existing'
        return re.sub(r'_existing$', '', stem)
    # Parent dir name carries the label
    parent = path.parent.name  # e.g. 'eq_gptoss', 'bm_deepseek', 'gptoss_120b'
    # Strip leading 'eq_' or 'bm_' prefixes from capture sub-dirs
    label = re.sub(r'^(eq|bm)_', '', parent)
    return label


# ── Load every JSONL file under bedrock/ ─────────────────────────────────────
def load_all_traces() -> pd.DataFrame:
    rows = []

    # 1. Generic results.jsonl files (bedrock/**/results.jsonl)
    for path_str in glob.glob(str(BEDROCK / "**" / "results.jsonl"), recursive=True):
        path = Path(path_str)
        label = _model_label(path)
        try:
            with open(path) as f:
                for line in f:
                    line = line.strip()
                    if not line:
                        continue
                    rec = json.loads(line)
                    rec["model"] = label
                    rec["source_file"] = str(path.relative_to(REPO))
                    rows.append(rec)
        except Exception as e:
            print(f"  [warn] could not read {path}: {e}")

    # 2. Special deepseek_existing.jsonl (equation_numeric_guess corrects)
    ds_path = BEDROCK / "eqguess_capture" / "deepseek_existing.jsonl"
    if ds_path.exists():
        label = _model_label(ds_path)
        try:
            with open(ds_path) as f:
                for line in f:
                    line = line.strip()
                    if not line:
                        continue
                    rec = json.loads(line)
                    rec["model"] = label
                    rec["source_file"] = str(ds_path.relative_to(REPO))
                    rows.append(rec)
        except Exception as e:
            print(f"  [warn] could not read {ds_path}: {e}")

    if not rows:
        raise RuntimeError("No trace records found — check BEDROCK path.")

    df = pd.DataFrame(rows)

    # Normalise column types
    for col in ["correct", "has_box"]:
        if col in df.columns:
            df[col] = df[col].astype(bool)
    for col in ["out_tokens", "reasoning_chars"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # text is present only for correct-and-saved traces; normalise None/NaN → ""
    if "text" in df.columns:
        df["text"] = df["text"].fillna("")
    else:
        df["text"] = ""

    df["has_text"] = df["text"].str.len() > 0

    # ── Join problem prompts from train.csv ───────────────────────────────────
    if "prompt" not in df.columns:
        df["prompt"] = ""

    if TRAIN_CSV.exists():
        train = pd.read_csv(TRAIN_CSV, dtype={"id": str})
        # id in train.csv may be zero-padded 8-char hex; normalise
        train["id"] = train["id"].str.strip()
        df["id"] = df["id"].astype(str).str.strip()
        prompt_map = train.set_index("id")["prompt"].to_dict()
        missing_mask = df["prompt"].isna() | (df["prompt"] == "")
        df.loc[missing_mask, "prompt"] = df.loc[missing_mask, "id"].map(prompt_map)
    else:
        print(f"[warn] train.csv not found at {TRAIN_CSV}; prompts will be empty.")

    return df.reset_index(drop=True)


print("Loading traces …")
df = load_all_traces()
print(f"Loaded {len(df):,} records across {df['model'].nunique()} models.")
print(f"Categories : {sorted(df['category'].dropna().unique())}")
print(f"Records with trace text: {df['has_text'].sum():,}")
df.head(3)

Loading traces …


Loaded 62,003 records across 14 models.
Categories : ['bit_manipulation', 'cryptarithm_deduce', 'cryptarithm_guess', 'equation_numeric_guess']
Records with trace text: 3,052


,id,gen,category,status,correct,extracted,answer,has_box,out_tokens,reasoning_chars,stop,model,source_file,text,answer_text,has_text,prompt
0,00c032a8,1,cryptarithm_deduce,rule_unknown,False,#?,\^?,True,2100,0,end_turn,_test_deepseek,bedrock/_test_deepseek/results.jsonl,,NaN,False,"In Alice's Wonderland, a secret set of transfo..."
1,00457d26,0,cryptarithm_deduce,rule_unknown,False,[!@,@&,True,2403,0,end_turn,_test_deepseek,bedrock/_test_deepseek/results.jsonl,,NaN,False,"In Alice's Wonderland, a secret set of transfo..."
2,00c032a8,0,cryptarithm_deduce,rule_unknown,False,#?,\^?,True,2449,0,end_turn,_test_deepseek,bedrock/_test_deepseek/results.jsonl,,NaN,False,"In Alice's Wonderland, a secret set of transfo..."


In [2]:
# ── Cell 2: SUMMARY TABLE ────────────────────────────────────────────────────
# Per (model, category): total records, # correct, # correct WITH trace saved.

summary = (
    df.groupby(["model", "category"])
    .agg(
        total=("id", "count"),
        correct=("correct", "sum"),
        correct_with_trace=("has_text", lambda s: (s & df.loc[s.index, "correct"]).sum()),
    )
    .reset_index()
)

summary["pct_correct"] = (summary["correct"] / summary["total"] * 100).round(1)

print("Per-model, per-category trace summary")
print("=" * 70)
display(
    summary.sort_values(["category", "model"])
    [["category", "model", "total", "correct", "pct_correct", "correct_with_trace"]]
    .rename(columns={"pct_correct": "% correct", "correct_with_trace": "w/ trace"})
    .set_index(["category", "model"])
)

Per-model, per-category trace summary


total  correct  % correct  \
category               model                                             
bit_manipulation       deepseek               3600      706       19.6   
                       deepseek_v3p2          1219      270       22.1   
                       glm                    3412      419       12.3   
                       gptoss                 3600      631       17.5   
                       gptoss_120b            2400      405       16.9   
                       mistral                3600      609       16.9   
                       mistral_large3         2400      420       17.5   
                       qwen                   2200      126        5.7   
cryptarithm_deduce     _test_deepseek            4        0        0.0   
                       deepseek_v3p2          2400        6        0.2   
                       glm5                   2400        2        0.1   
                       gptoss_120b            2400        1        0.0   
                       gptoss_120b_val           3        0        0.0   
                       kimi_k2p5              2400        1        0.0   
                       mistral_large3         2400        4        0.2   
                       nemotron_super3_120b   2400        0        0.0   
                       qwen3_235b             2400        1        0.0   
cryptarithm_guess      deepseek_v3p2          2400        2        0.1   
                       glm5                   1757        0        0.0   
                       gptoss_120b            2400        9        0.4   
                       kimi_k2p5              2400        0        0.0   
                       mistral_large3         2400        1        0.0   
                       nemotron_super3_120b   2400        0        0.0   
                       qwen3_235b              918        1        0.1   
equation_numeric_guess deepseek                135      135      100.0   
                       deepseek_v3p2          2176      135        6.2   
                       gptoss                   51        0        0.0   
                       gptoss_120b            2176      197        9.1   
                       kimi_k2p5               430        5        1.2   
                       mistral                 104        0        0.0   
                       mistral_large3         2176      120        5.5   
                       nemotron_super3_120b    842       12        1.4   

                                             w/ trace  
category               model                           
bit_manipulation       deepseek                   706  
                       deepseek_v3p2              270  
                       glm                        419  
                       gptoss                     631  
                       gptoss_120b                  0  
                       mistral                    609  
                       mistral_large3               0  
                       qwen                       126  
cryptarithm_deduce     _test_deepseek               0  
                       deepseek_v3p2                0  
                       glm5                         1  
                       gptoss_120b                  0  
                       gptoss_120b_val              0  
                       kimi_k2p5                    0  
                       mistral_large3               0  
                       nemotron_super3_120b         0  
                       qwen3_235b                   0  
cryptarithm_guess      deepseek_v3p2                2  
                       glm5                         0  
                       gptoss_120b                  0  
                       kimi_k2p5                    0  
                       mistral_large3               0  
                       nemotron_super3_120b         0  
                       qwen3_235b                   1  
equation_numeric_guess deepseek                   135  
                

In [3]:
# ── Cell 3: BROWSER FUNCTION ─────────────────────────────────────────────────
# show(category, model, correct, only_with_text, n, full)
#   Prints a clean, human-readable view of each matching trace.

def show(
    category: str | None = None,
    model: str | None = None,
    correct: bool = True,
    only_with_text: bool = True,
    n: int = 5,
    full: bool = False,
):
    """
    Browse captured reasoning traces.

    Parameters
    ----------
    category      : filter by category substring (e.g. 'equation', 'bit')
    model         : filter by model name substring (e.g. 'deepseek', 'gptoss')
    correct       : if True, show only correct answers (default True)
    only_with_text: if True, show only records that have the trace text saved
    n             : max number of records to display
    full          : if True, print the complete trace; otherwise truncate to
                    2 000 chars for the reasoning + full answer_text
    """
    mask = pd.Series(True, index=df.index)

    if category is not None:
        mask &= df["category"].str.contains(category, case=False, na=False)
    if model is not None:
        mask &= df["model"].str.contains(model, case=False, na=False)
    if correct:
        mask &= df["correct"].fillna(False)
    if only_with_text:
        mask &= df["has_text"]

    subset = df[mask].sample(frac=1, random_state=42).head(n)  # random sample

    if subset.empty:
        print("No records match the given filters.")
        return

    SEP  = "═" * 80
    SEP2 = "─" * 80
    TRACE_MAX = 2_000  # chars to show in truncated mode

    for i, (_, row) in enumerate(subset.iterrows(), 1):
        print(SEP)
        print(f"  [{i}/{len(subset)}]  id={row['id']}  model={row['model']}  "
              f"category={row.get('category','?')}  status={row.get('status','?')}")
        print(SEP)

        # PROMPT
        prompt_text = str(row.get("prompt") or "(prompt not available)")
        print("PROMPT")
        print(SEP2)
        # Wrap long prompts for readability
        for line in prompt_text.split("\n"):
            print(textwrap.fill(line, width=78) if len(line) > 78 else line)
        print()

        # GROUND-TRUTH vs MODEL ANSWER
        print(f"GROUND-TRUTH answer : {row.get('answer', '?')}")
        print(f"Model extracted     : {row.get('extracted', '?')}")
        print(f"Answer text (model) : {str(row.get('answer_text', ''))[:300]}")
        print()

        # TRACE
        trace = str(row.get("text") or "")
        print("REASONING TRACE")
        print(SEP2)
        if trace:
            if full or len(trace) <= TRACE_MAX:
                print(trace)
            else:
                half = TRACE_MAX // 2
                print(trace[:half])
                print(f"\n … [{len(trace) - TRACE_MAX:,} chars omitted] … "
                      "(pass full=True to see everything)\n")
                print(trace[-half:])
        else:
            print("(trace text not saved for this record)")
        print()

    print(SEP)
    print(f"Showed {len(subset)} record(s). "
          f"Total matching: {mask.sum():,}.")


print("show() function defined. Usage example:")
print("  show(category='equation', model='deepseek', n=3)")
print("  show(category='bit', model='gptoss', n=2, full=True)")

show() function defined. Usage example:
  show(category='equation', model='deepseek', n=3)
  show(category='bit', model='gptoss', n=2, full=True)


In [4]:
# ── Cell 4: EXAMPLE — correct equation_numeric_guess traces (deepseek) ───────
# Shows 3 randomly-sampled correct traces from the deepseek capture file.
# Edit model= or n= to explore other models.

print("=== Correct equation_numeric_guess traces (deepseek) ===")
show(category="equation_numeric_guess", model="deepseek", correct=True, only_with_text=True, n=3)

=== Correct equation_numeric_guess traces (deepseek) ===
════════════════════════════════════════════════════════════════════════════════
  [1/3]  id=ea6d926a  model=deepseek_v3p2  category=equation_numeric_guess  status=rule_unknown
════════════════════════════════════════════════════════════════════════════════
PROMPT
────────────────────────────────────────────────────────────────────────────────
In Alice's Wonderland, a secret set of transformation rules is applied to
equations. Below are a few examples:
15-77 = 62
78*32 = 2002
41-05 = 8
95*11 = 056
Now, determine the result for: 25+62

GROUND-TRUTH answer : 87
Model extracted     : 87
Answer text (model) : Let’s analyze the examples carefully.  

---

**Step 1: Observe given equations**

1. \( 15 - 77 = 62 \)  
   Normally \( 15 - 77 = -62 \), but here it’s \( 62 \) (positive).  
   Suggests maybe: \( |15 - 77| = 62 \) — but that’s still \( 62 \), same as RHS.

2. \( 78 * 32 = 2002 \)  
   Normally \(

REASONING TRACE
────────────

In [5]:
# ── Cell 5: EXAMPLE — correct bit_manipulation traces (gptoss) ───────────────
# Shows 3 randomly-sampled correct bit_manipulation traces.

print("=== Correct bit_manipulation traces (gptoss) ===")
show(category="bit_manipulation", model="gptoss", correct=True, only_with_text=True, n=3)

=== Correct bit_manipulation traces (gptoss) ===
════════════════════════════════════════════════════════════════════════════════
  [1/3]  id=06120e47  model=gptoss  category=bit_manipulation  status=hypothesis_formed
════════════════════════════════════════════════════════════════════════════════
PROMPT
────────────────────────────────────────────────────────────────────────────────
In Alice's Wonderland, a secret bit manipulation rule transforms 8-bit binary
numbers. The transformation involves operations like bit shifts, rotations,
XOR, AND, OR, NOT, and possibly majority or choice functions.

Here are some examples of input -> output:
01010100 -> 11101111
10000011 -> 11111101
11110110 -> 11100101
10010111 -> 11111001
10101101 -> 11011111
00111010 -> 11110111
11101111 -> 11000111
00100110 -> 11110111
01001000 -> 11111110

Now, determine the output for: 01110011

GROUND-TRUTH answer : 11110010
Model extracted     : 11111111
Answer text (model) : \boxed{11111111}

REASONING TRACE
────

In [6]:
# ── Cell 6: ANSWER-DISTRIBUTION ANALYSIS (optional analytical aid) ──────────
# For equation_numeric_guess: the puzzle gives an equation like
#   "A OP B = ?"  where OP is +, -, *, //, etc.
# This cell checks — for correct guesses — whether the model's extracted
# answer commonly coincides with simple functions of the two operands
# (e.g. |A-B|, A+B, A*B …), acting as a quick sanity check / insight tool.

import re
import numpy as np

TARGET_CATEGORY = "equation_numeric_guess"

eq_correct = df[
    (df["category"] == TARGET_CATEGORY) &
    df["correct"].fillna(False)
].copy()

print(f"Correct {TARGET_CATEGORY} records: {len(eq_correct)}")

# Try to parse two integer operands from the prompt (heuristic: first two
# standalone integers after the word 'equation' or in an 'A OP B' pattern).
def extract_operands(prompt: str):
    """Return (A, B) if two operand integers can be found, else None."""
    if not isinstance(prompt, str):
        return None
    # Look for patterns like  "12 + 34"  or  "12 - 34"  or  "equation: 12 * 34"
    m = re.search(
        r'\b(-?\d+)\s*[+\-*/÷×]\s*(-?\d+)\b',
        prompt
    )
    if m:
        return int(m.group(1)), int(m.group(2))
    # Fallback: first two standalone integers in the prompt
    nums = re.findall(r'\b(\d{1,6})\b', prompt)
    if len(nums) >= 2:
        return int(nums[0]), int(nums[1])
    return None

eq_correct["operands"] = eq_correct["prompt"].apply(extract_operands)
parseable = eq_correct[eq_correct["operands"].notna()].copy()
print(f"Records with parseable operands: {len(parseable)}")

if len(parseable) > 0:
    parseable["A"] = parseable["operands"].apply(lambda t: t[0])
    parseable["B"] = parseable["operands"].apply(lambda t: t[1])
    parseable["ans"] = pd.to_numeric(parseable["answer"], errors="coerce")

    # Candidate operations to check
    candidates = {
        "A+B":  lambda r: r.A + r.B,
        "A-B":  lambda r: r.A - r.B,
        "B-A":  lambda r: r.B - r.A,
        "|A-B|": lambda r: abs(r.A - r.B),
        "A*B":  lambda r: r.A * r.B,
        "A//B (if B≠0)": lambda r: r.A // r.B if r.B != 0 else np.nan,
        "A%B (if B≠0)":  lambda r: r.A %  r.B if r.B != 0 else np.nan,
        "max(A,B)": lambda r: max(r.A, r.B),
        "min(A,B)": lambda r: min(r.A, r.B),
    }

    results = {}
    for name, fn in candidates.items():
        try:
            vals = parseable.apply(fn, axis=1)
            matches = (vals == parseable["ans"]).sum()
            results[name] = matches
        except Exception:
            results[name] = 0

    dist = pd.Series(results).sort_values(ascending=False)
    dist_pct = (dist / len(parseable) * 100).round(1)
    dist_df = pd.DataFrame({"matches": dist, "% of parseable": dist_pct})
    dist_df.index.name = "operation"

    print(f"\nAnswer-distribution for correct {TARGET_CATEGORY} "
          f"(n={len(parseable)} parseable):")
    display(dist_df)
else:
    print("Could not parse operands from prompts — skipping distribution analysis.")

Correct equation_numeric_guess records: 604
Records with parseable operands: 604

Answer-distribution for correct equation_numeric_guess (n=604 parseable):


,matches,% of parseable
operation,,
A-B,75,12.4
|A-B|,70,11.6
A%B (if B≠0),10,1.7
B-A,3,0.5
A+B,0,0.0
A*B,0,0.0
A//B (if B≠0),0,0.0
"max(A,B)",0,0.0
"min(A,B)",0,0.0
